In [ ]:
#TEST PAGE
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time

# 1. Purpose: Setup Chrome driver 
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),
                          options=options)

# 2. Purpose: Link to Reddit site
page_url = "https://www.reddit.com/r/AskReddit/comments/1fwmotf/do_long_distance_relationships_work_whats_your/"

driver.get(page_url)

# 3. Purpose: Pause so the page + reviews section can load
time.sleep(5)

In [ ]:
#TEST PAGE
# Purpose: Find all comment elements on the page
comments = driver.find_elements(By.TAG_NAME, "shreddit-comment")

print(f"Number of comments found: {len(comments)}")

organized_data = []

for i, comment in enumerate(comments):
    try:
        # 1. Extract Username
        # Reddit stores this in the 'author' attribute of the shreddit-comment tag
        username = comment.get_attribute("author")
        
        # 2. Extract Content
        # look for the div with slot="comment" which holds the text
        try:
            # We use a relative CSS selector to find the text div inside this specific comment
            content_element = comment.find_element(By.CSS_SELECTOR, "div[slot='comment']")
            content = content_element.text.replace('\n', ' ').strip()
        except:
            content = "Content hidden or deleted"

        # 3. Display to check (Step-by-Step)
        if i < 5: # Only print first 5 to verify
            print(f"\n--- Comment {i+1} ---")
            print(f"User: {username}")
            print(f"Text: {content[:100]}...")

        organized_data.append({
            "Username": username,
            "Content": content
        })

    except Exception as e:
        print(f"Skipping a comment due to error: {e}")
        continue

Number of comments found: 24

--- Comment 1 ---
User: Poltira
Text: Loads of people in this sub closed the gap, loads of people are making LDR work perfectly, and loads...

--- Comment 2 ---
User: AverageJimbo99
Text: It works when you both agree to eventually close the gap and even better when you set a timeframe fo...

--- Comment 3 ---
User: User_reddit__
Text: It does work in a very rare case mine didn’t....

--- Comment 4 ---
User: SirLunchALot1993
Text: It really depends on the people. If you are super needy, clingy and need lots of lots of phsyical To...

--- Comment 5 ---
User: [deleted]
Text: Content hidden or deleted...


In [1]:
import csv
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# 1. Setup Driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. Open Reddit Thread
url = "https://www.reddit.com/r/AskReddit/comments/1fwmotf/do_long_distance_relationships_work_whats_your/"
driver.get(url)
time.sleep(5)

# 3. Purpose: Scroll to load more comments
# should scroll 3 times to get a larger sample size (roughly 50-100 comments)
for i in range(3):
    print(f"Scrolling to load more content... ({i+1}/3)")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(4)

# 4. Purpose: Extract all comments
comment_elements = driver.find_elements(By.TAG_NAME, "shreddit-comment")
print(f"\nTotal comments found: {len(comment_elements)}")

organized_data = []

for comment in comment_elements:
    try:
        # Extract Username from 'author' attribute
        username = comment.get_attribute("author")
        
        # Extract Content from the specific comment slot
        try:
            content_element = comment.find_element(By.CSS_SELECTOR, "div[slot='comment']")
            content = content_element.text.replace('\n', ' ').strip()
        except:
            content = "[Deleted/Hidden]"

        # Only add if we have a valid username and content
        if username and content != "[Deleted/Hidden]":
            organized_data.append({
                "Username": username,
                "Content": content
            })
    except:
        continue

# 5. Purpose: Save to CSV
filename = "reddit_reviews.csv"
with open(filename, 'w', newline='', encoding='utf-8') as f:
    # Defining the columns for the CSV
    writer = csv.DictWriter(f, fieldnames=["Username", "Content"])
    writer.writeheader()
    writer.writerows(organized_data)

print(f"\nSUCCESS! {len(organized_data)} rows saved to '{filename}'.")

driver.quit()

Scrolling to load more content... (1/3)
Scrolling to load more content... (2/3)
Scrolling to load more content... (3/3)

Total comments found: 24

SUCCESS! 23 rows saved to 'reddit_reviews.csv'.


In [2]:
import csv
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# 1. Setup Driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. List of Reddit threads to scrape
urls = [
    "https://www.reddit.com/r/AskWomenOver30/comments/1ig7r53/please_tell_me_your_experiences_with_long/",
    "https://www.reddit.com/r/AskWomen/comments/1kkg4j6/anyone_who_has_started_a_relationship_long/",
]

filename = "reddit_reviews.csv"

# Loop through each URL one by one
for page_url in urls:
    print(f"\n--- Starting Extraction for: {page_url} ---")
    driver.get(page_url)
    time.sleep(5) # Initial load wait

    # 3. Purpose: Scroll to load more content for THIS specific page
    for i in range(10): 
        print(f"Scrolling page... ({i+1}/10)")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(4)

    # 4. Purpose: Extract all comments on the current page
    comment_elements = driver.find_elements(By.TAG_NAME, "shreddit-comment")
    print(f"Found {len(comment_elements)} total comments on this thread.")

    organized_data = []

    for comment in comment_elements:
        try:
            # Extract Username
            username = comment.get_attribute("author")
            
            # Extract Content from the specific comment slot
            try:
                content_element = comment.find_element(By.CSS_SELECTOR, "div[slot='comment']")
                content = content_element.text.replace('\n', ' ').strip()
            except:
                content = "[Deleted/Hidden]"

            # Only add if we have valid data
            if username and content != "[Deleted/Hidden]":
                organized_data.append({
                    "Username": username,
                    "Content": content
                })
        except:
            continue

    # 5. Purpose: Save to CSV (Append Mode)
    # We use 'a' so it doesn't delete data from the previous URL
    file_exists = os.path.isfile(filename)
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["Username", "Content"])
        if not file_exists:
            writer.writeheader() # Only write header once
            file_exists = True # Update flag so next URL doesn't write header
        writer.writerows(organized_data)

    print(f"Finished. Added {len(organized_data)} rows to {filename}.")

# Close the browser after all URLs are done
print("\n--- ALL URLS PROCESSED ---")
driver.quit()


--- Starting Extraction for: https://www.reddit.com/r/AskWomenOver30/comments/1ig7r53/please_tell_me_your_experiences_with_long/ ---
Scrolling page... (1/10)
Scrolling page... (2/10)
Scrolling page... (3/10)
Scrolling page... (4/10)
Scrolling page... (5/10)
Scrolling page... (6/10)
Scrolling page... (7/10)
Scrolling page... (8/10)
Scrolling page... (9/10)
Scrolling page... (10/10)
Found 15 total comments on this thread.
Finished. Added 15 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/AskWomen/comments/1kkg4j6/anyone_who_has_started_a_relationship_long/ ---
Scrolling page... (1/10)
Scrolling page... (2/10)
Scrolling page... (3/10)
Scrolling page... (4/10)
Scrolling page... (5/10)
Scrolling page... (6/10)
Scrolling page... (7/10)
Scrolling page... (8/10)
Scrolling page... (9/10)
Scrolling page... (10/10)
Found 76 total comments on this thread.
Finished. Added 74 rows to reddit_reviews.csv.

--- ALL URLS PROCESSED ---


In [3]:
import csv
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# 1. Setup Driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. List of Reddit threads to scrape
urls = [
    "https://www.reddit.com/r/dating/comments/1ft8sly/do_you_guys_believe_in_long_distance_relationship/",
    "https://www.reddit.com/r/AskWomen/comments/1ltcl5f/what_was_your_experience_with_longdistance_love/",
    "https://www.reddit.com/r/UniUK/comments/1cfxwse/share_your_longdistance_relationship_stories/"
]

filename = "reddit_reviews.csv"

# Loop through each URL one by one
for page_url in urls:
    print(f"\n--- Starting Extraction for: {page_url} ---")
    driver.get(page_url)
    time.sleep(5) # Initial load wait

    # 3. Purpose: Scroll to load more content for THIS specific page
    for i in range(10): 
        print(f"Scrolling page... ({i+1}/10)")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(4)

    # 4. Purpose: Extract all comments on the current page
    comment_elements = driver.find_elements(By.TAG_NAME, "shreddit-comment")
    print(f"Found {len(comment_elements)} total comments on this thread.")

    organized_data = []

    for comment in comment_elements:
        try:
            # Extract Username
            username = comment.get_attribute("author")
            
            # Extract Content from the specific comment slot
            try:
                content_element = comment.find_element(By.CSS_SELECTOR, "div[slot='comment']")
                content = content_element.text.replace('\n', ' ').strip()
            except:
                content = "[Deleted/Hidden]"

            # Only add if we have valid data
            if username and content != "[Deleted/Hidden]":
                organized_data.append({
                    "Username": username,
                    "Content": content
                })
        except:
            continue

    # 5. Purpose: Save to CSV (Append Mode)
    # We use 'a' so it doesn't delete data from the previous URL
    file_exists = os.path.isfile(filename)
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["Username", "Content"])
        if not file_exists:
            writer.writeheader() # Only write header once
            file_exists = True # Update flag so next URL doesn't write header
        writer.writerows(organized_data)

    print(f"Finished. Added {len(organized_data)} rows to {filename}.")

# Close the browser after all URLs are done
print("\n--- ALL URLS PROCESSED ---")
driver.quit()


--- Starting Extraction for: https://www.reddit.com/r/dating/comments/1ft8sly/do_you_guys_believe_in_long_distance_relationship/ ---
Scrolling page... (1/10)
Scrolling page... (2/10)
Scrolling page... (3/10)
Scrolling page... (4/10)
Scrolling page... (5/10)
Scrolling page... (6/10)
Scrolling page... (7/10)
Scrolling page... (8/10)
Scrolling page... (9/10)
Scrolling page... (10/10)
Found 63 total comments on this thread.
Finished. Added 62 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/AskWomen/comments/1ltcl5f/what_was_your_experience_with_longdistance_love/ ---
Scrolling page... (1/10)
Scrolling page... (2/10)
Scrolling page... (3/10)
Scrolling page... (4/10)
Scrolling page... (5/10)
Scrolling page... (6/10)
Scrolling page... (7/10)
Scrolling page... (8/10)
Scrolling page... (9/10)
Scrolling page... (10/10)
Found 52 total comments on this thread.
Finished. Added 49 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/U

In [4]:
import csv
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# 1. Setup Driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. List of Reddit threads to scrape
urls = [
    "https://www.reddit.com/r/LongDistance/comments/1fx5obm/tell_me_about_your_ldr/",
    "https://www.reddit.com/r/AskReddit/comments/1j83m31/what_are_your_thoughts_on_long_distance/",
    "https://www.reddit.com/r/LongDistance/comments/10p6362/what_are_your_opinions_on_this_when_it_comes_to/"
]

filename = "reddit_reviews.csv"

# Loop through each URL one by one
for page_url in urls:
    print(f"\n--- Starting Extraction for: {page_url} ---")
    driver.get(page_url)
    time.sleep(5) # Initial load wait

    # 3. Purpose: Scroll to load more content for THIS specific page
    for i in range(10): 
        print(f"Scrolling page... ({i+1}/10)")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(4)

    # 4. Purpose: Extract all comments on the current page
    comment_elements = driver.find_elements(By.TAG_NAME, "shreddit-comment")
    print(f"Found {len(comment_elements)} total comments on this thread.")

    organized_data = []

    for comment in comment_elements:
        try:
            # Extract Username
            username = comment.get_attribute("author")
            
            # Extract Content from the specific comment slot
            try:
                content_element = comment.find_element(By.CSS_SELECTOR, "div[slot='comment']")
                content = content_element.text.replace('\n', ' ').strip()
            except:
                content = "[Deleted/Hidden]"

            # Only add if we have valid data
            if username and content != "[Deleted/Hidden]":
                organized_data.append({
                    "Username": username,
                    "Content": content
                })
        except:
            continue

    # 5. Purpose: Save to CSV (Append Mode)
    # We use 'a' so it doesn't delete data from the previous URL
    file_exists = os.path.isfile(filename)
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["Username", "Content"])
        if not file_exists:
            writer.writeheader() # Only write header once
            file_exists = True # Update flag so next URL doesn't write header
        writer.writerows(organized_data)

    print(f"Finished. Added {len(organized_data)} rows to {filename}.")

# Close the browser after all URLs are done
print("\n--- ALL URLS PROCESSED ---")
driver.quit()


--- Starting Extraction for: https://www.reddit.com/r/LongDistance/comments/1fx5obm/tell_me_about_your_ldr/ ---
Scrolling page... (1/10)
Scrolling page... (2/10)
Scrolling page... (3/10)
Scrolling page... (4/10)
Scrolling page... (5/10)
Scrolling page... (6/10)
Scrolling page... (7/10)
Scrolling page... (8/10)
Scrolling page... (9/10)
Scrolling page... (10/10)
Found 24 total comments on this thread.
Finished. Added 23 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/AskReddit/comments/1j83m31/what_are_your_thoughts_on_long_distance/ ---
Scrolling page... (1/10)
Scrolling page... (2/10)
Scrolling page... (3/10)
Scrolling page... (4/10)
Scrolling page... (5/10)
Scrolling page... (6/10)
Scrolling page... (7/10)
Scrolling page... (8/10)
Scrolling page... (9/10)
Scrolling page... (10/10)
Found 84 total comments on this thread.
Finished. Added 84 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/LongDistance/comments/10p6362

In [6]:
import csv
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# 1. Setup Driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. List of Reddit threads to scrape
urls = [
    "https://www.reddit.com/r/LongDistance/comments/x1qgfq/what_is_the_most_helpful_thing_to_sustain_a_long/",
    "https://www.reddit.com/r/LDR/comments/1ojd2e0/thoughts_on_starting_a_relationship_that_is/",
    "https://www.reddit.com/r/AskReddit/comments/1qxr5gd/whats_everyones_thoughts_on_long_distance/"
]

filename = "reddit_reviews.csv"

# Loop through each URL one by one
for page_url in urls:
    print(f"\n--- Starting Extraction for: {page_url} ---")
    driver.get(page_url)
    time.sleep(5) # Initial load wait

    # 3. Purpose: Scroll to load more content for THIS specific page
    for i in range(5): 
        print(f"Scrolling page... ({i+1}/5)")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(4)

    # 4. Purpose: Extract all comments on the current page
    comment_elements = driver.find_elements(By.TAG_NAME, "shreddit-comment")
    print(f"Found {len(comment_elements)} total comments on this thread.")

    organized_data = []

    for comment in comment_elements:
        try:
            # Extract Username
            username = comment.get_attribute("author")
            
            # Extract Content from the specific comment slot
            try:
                content_element = comment.find_element(By.CSS_SELECTOR, "div[slot='comment']")
                content = content_element.text.replace('\n', ' ').strip()
            except:
                content = "[Deleted/Hidden]"

            # Only add if we have valid data
            if username and content != "[Deleted/Hidden]":
                organized_data.append({
                    "Username": username,
                    "Content": content
                })
        except:
            continue

    # 5. Purpose: Save to CSV (Append Mode)
    # We use 'a' so it doesn't delete data from the previous URL
    file_exists = os.path.isfile(filename)
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["Username", "Content"])
        if not file_exists:
            writer.writeheader() # Only write header once
            file_exists = True # Update flag so next URL doesn't write header
        writer.writerows(organized_data)

    print(f"Finished. Added {len(organized_data)} rows to {filename}.")

# Close the browser after all URLs are done
print("\n--- ALL URLS PROCESSED ---")
driver.quit()


--- Starting Extraction for: https://www.reddit.com/r/LongDistance/comments/x1qgfq/what_is_the_most_helpful_thing_to_sustain_a_long/ ---
Scrolling page... (1/5)
Scrolling page... (2/5)
Scrolling page... (3/5)
Scrolling page... (4/5)
Scrolling page... (5/5)
Found 34 total comments on this thread.
Finished. Added 34 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/LDR/comments/1ojd2e0/thoughts_on_starting_a_relationship_that_is/ ---
Scrolling page... (1/5)
Scrolling page... (2/5)
Scrolling page... (3/5)
Scrolling page... (4/5)
Scrolling page... (5/5)
Found 12 total comments on this thread.
Finished. Added 12 rows to reddit_reviews.csv.

--- Starting Extraction for: https://www.reddit.com/r/AskReddit/comments/1qxr5gd/whats_everyones_thoughts_on_long_distance/ ---
Scrolling page... (1/5)
Scrolling page... (2/5)
Scrolling page... (3/5)
Scrolling page... (4/5)
Scrolling page... (5/5)
Found 23 total comments on this thread.
Finished. Added 23 rows to reddit_